In [1]:
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import logging
import time
import os
from tqdm import tqdm
from polar_bimamba_model import PolarBiMambaDecoder
from polar_bimamba_dataset import BER, FER, bin_to_sign
from polar_bimamba_init import initialize, save_checkpoint, dump_config



/home/aayush/Desktop/5G-Polar/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
PARAMETERS = dict(
    code_hint="POLAR_N32_K16",
    d_model=128,                
    d_state=64,                 
    d_conv=6,                   
    expand=2,                 
    N_dec=12,                   
    warmup_lr=1e-3,             
    warmup_length=20,           
    epochs=500,                 
    T_max=100,                  
    train_batch_count=500,      
    batch_size=64,             
    test_batch_count=500,       
    test_batch_size=32,         
    lr=1e-4,                   
    eta_min=1e-7,#scheduler
    gradient_clipping=1.0,      
    dropout=0.1,                
    seq_len=32,                
    enable_multi_loss=True,     
    enable_early_stopping=True,
    early_stopping_patience=3,  
    zero_cw=False,              
    workers=0                  
)


In [3]:
# #LOAD MODEL
# from main.polar_bimamba_init import load_checkpoint

# PREV_EXP_PATH = "./bimamba_results/AB3DE16DE6D7981A6FA0BAC296CB5574"  # previous folder
# OUTPUT_PATH = "./bimamba_results/resume_training"  # new folder 
# os.makedirs(OUTPUT_PATH, exist_ok=True)


# checkpoint = load_checkpoint(PREV_EXP_PATH)
# config = checkpoint['config']

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = PolarBiMambaDecoder(config).to(device)

# best_model_path = os.path.join(PREV_EXP_PATH, "best_model.pt")
# if os.path.exists(best_model_path):
#     model.load_state_dict(torch.load(best_model_path, map_location=device))
#     print(f"Loaded best model from {best_model_path}")
# else:
#     model.load_state_dict(checkpoint['model'])
#     print("Loaded last saved model from checkpoint")


# config.path = OUTPUT_PATH  # save checkpoints in new folder

# config, model, optimizer, training_state, dataset_tuple, summary_writer = initialize(
#     path=OUTPUT_PATH,
#     model_cls=PolarBiMambaDecoder,
#     **PARAMETERS
# )

# train_loader, test_loader_list, _, EbNo_range_test = dataset_tuple
# summary_writer = SummaryWriter(log_dir=OUTPUT_PATH)



In [4]:
def update_training_state(training_state, epoch, loss, ber):
    training_state['epoch'] = epoch
    training_state['loss'] = loss
    training_state['BER'] = ber
    if ber < training_state.get('best_ber', float('inf')):
        training_state['best_ber'] = ber
        training_state['best_ber_epoch'] = epoch
    if loss < training_state.get('best_loss', float('inf')):
        training_state['best_loss'] = loss
        training_state['best_loss_epoch'] = epoch
    return training_state



In [5]:
def update_test_state(test_state, results, epoch):
    test_state.update(results)
    for key in results:
        best_key = f'best_{key}'
        best_result = test_state.get(best_key, float('inf'))
        if best_result <= results[key]:
            continue
        test_state[best_key] = results[key]
        test_state[f'{best_key}_epoch'] = epoch
    return test_state

In [6]:
def epoch_callback(config, training_state, model, optimizer, summary_writer, **kwargs):
    checkpoint = {
        'config': config,
        'state': training_state,
        'optimizer': optimizer.state_dict(),
        'model': model.state_dict()
    }
    # Save best model based on loss
    if training_state['best_loss'] <= training_state['loss']:
        checkpoint['best_model'] = model.state_dict()
        torch.save(model.state_dict(), os.path.join(config.path, 'best_model.pt'))
    save_checkpoint(checkpoint)
    
    # TensorBoard logging
    summary_writer.add_scalar('Train: Loss/Epoch', training_state['loss'], training_state['epoch'])
    summary_writer.add_scalar('Train: BER/Epoch', training_state['BER'], training_state['epoch'])
    summary_writer.add_scalar('Train: Best Loss/Epoch', training_state['best_loss'], training_state['epoch'])
    summary_writer.add_scalar('Train: Best BER/Epoch', training_state['best_ber'], training_state['epoch'])
    
    if (scheduler := kwargs.get('scheduler')):
        summary_writer.add_scalar('Train: LR/Epoch', scheduler.get_last_lr()[0], training_state['epoch'])



In [7]:
def test_callback(config, training_state, test_results, summary_writer, model, **kwargs):
    run_name = os.path.basename(os.path.normpath(config.path))
    hparams_dir = os.path.join(config.path, run_name)
    if os.path.exists(hparams_dir):
        for filename in os.listdir(hparams_dir):
            os.remove(os.path.join(hparams_dir, filename))
    
    for key, value in test_results.items():
        if 'BER' in key and not key.endswith('epoch'):
            ln_ber = -np.log(value) if value > 0 else float('inf')
            summary_writer.add_scalar(f'Test: -ln({key})/Epoch', ln_ber, training_state['epoch'])
        else:
            summary_writer.add_scalar(f'Test: {key}/Epoch', value, training_state['epoch'])
        # Save best model for each metric
        if (best_result := test_results.get(f'best_{key}')) is not None and best_result >= value:
            torch.save(model.state_dict(), os.path.join(config.path, f'best_model_{key}.pt'))
    
    summary_writer.add_hparams(
        dump_config(config),
        {**training_state, **test_results},
        run_name=run_name,
        global_step=training_state['epoch']
    )

In [8]:
def train_epoch(model, device, train_loader, optimizer, epoch, lr, config):
    model.train()
    cum_loss = cum_ber = cum_samples = 0.0
    t0 = time.time()
    for batch_idx, (m, x, z, y, magnitude, syndrome) in enumerate(tqdm(train_loader)):
        z_mul = y * bin_to_sign(x)
        z_pred = model(magnitude.to(device), syndrome.to(device))
        # loss without multi_loss keyword
        loss, x_pred = model.loss(z_pred, z_mul.to(device), y.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clipping)
        optimizer.step()
        
        ber = BER(x_pred, x.to(device))
        cum_loss += loss.item() * x.shape[0]
        cum_ber += ber * x.shape[0]
        cum_samples += x.shape[0]
    
    logging.info(f'Epoch {epoch} Train Loss={cum_loss/cum_samples:.2e}, BER={cum_ber/cum_samples:.2e}, Time={time.time()-t0:.2f}s')
    return cum_loss/cum_samples, cum_ber/cum_samples
# def train_epoch(model, device, train_loader, optimizer, epoch, lr, config):
#     model.train()
#     cum_loss = cum_ber = cum_samples = 0.0
#     t0 = time.time()
    
#     for batch_idx, (m, x, z, y, magnitude, syndrome) in enumerate(train_loader):
#         z_mul = y * bin_to_sign(x)
#         z_pred = model(magnitude.to(device), syndrome.to(device))
#         loss, x_pred = model.loss(z_pred, z_mul.to(device), y.to(device))

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clipping)
#         optimizer.step()

#         ber = BER(x_pred, x.to(device))
#         cum_loss += loss.item() * x.shape[0]
#         cum_ber += ber * x.shape[0]
#         cum_samples += x.shape[0]

#     logging.info(f'Epoch {epoch} Train Loss={cum_loss/cum_samples:.2e}, BER={cum_ber/cum_samples:.2e}, Time={time.time()-t0:.2f}s')
#     return cum_loss/cum_samples, cum_ber/cum_samples


In [ ]:
def test(model, device, test_loader_list, EbNo_range_test):
    model.eval()
    results = {}
    total_ber = 0
    with torch.no_grad():
        for ii, test_loader in enumerate(test_loader_list):
            test_ber = cum_count = 0
            for m, x, z, y, magnitude, syndrome in tqdm(test_loader):
                z_pred = model(magnitude.to(device), syndrome.to(device))
                x_pred = model.get_codeword(z_pred, y.to(device))
                test_ber += BER(x_pred, x.to(device)) * x.shape[0]
                cum_count += x.shape[0]
            test_ber /= cum_count
            results[f"BER_{EbNo_range_test[ii]}"] = test_ber
            total_ber += test_ber / len(test_loader_list)
    results['test_ber'] = total_ber
    return results




In [ ]:
def train_model(config, model, optimizer, training_state, dataset, summary_writer, epochs_per_test=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, test_loader_list, _, EbNo_range_test = dataset
    scheduler = None
    lr = config.warmup_lr
    test_state = {}
    
    for epoch in range(training_state.get('epoch', 0) + 1, config.epochs + 1):
        # Scheduler initialization
        if epoch >= config.warmup_length and scheduler is None:
            for pg in optimizer.param_groups:
                pg['lr'] = config.lr
                pg['initial_lr'] = config.lr 
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=config.T_max, eta_min=config.eta_min, last_epoch=epoch - config.warmup_length
            )
        if scheduler is not None:
            lr = scheduler.get_last_lr()[0]
        
        # Train
        loss, ber = train_epoch(model, device, train_loader, optimizer, epoch, lr, config)
        update_training_state(training_state, epoch, loss, ber)
        
        if scheduler is not None:
            scheduler.step()
        
        # Epoch callback
        epoch_callback(config, training_state, model, optimizer, summary_writer, scheduler=scheduler)
        
        # Test periodically
        if epoch % epochs_per_test == 0:
            results = test(model, device, test_loader_list, EbNo_range_test)
            update_test_state(test_state, results, epoch)
            test_callback(config, training_state, results, summary_writer, model, scheduler=scheduler)
            
            # --- Early stopping check ---
            if config.enable_early_stopping:
                current_ber = results['test_ber']  # use the correct key
                if 'best_ber' not in training_state:
                    training_state['best_ber'] = current_ber
                    training_state['patience_counter'] = 0
                elif current_ber < training_state['best_ber']:
                    training_state['best_ber'] = current_ber
                    training_state['patience_counter'] = 0
                else:
                    training_state['patience_counter'] += 1

                if training_state['patience_counter'] >= config.early_stopping_patience:
                    print(f"Early stopping triggered at epoch {epoch}")
                    break

    return model


In [11]:
output_path = "./bimamba_results"

config, model, optimizer, training_state, dataset, summary_writer = initialize(
    path=output_path,
    model_cls=PolarBiMambaDecoder,
    **PARAMETERS
)

Model initialized. Parameters: 3703457. N_dec=12, d_model=128, code=Code(n=32, k=16, code_type='POLAR', pc_matrix=tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,


In [12]:
trained_model = train_model(
    config,
    model,
    optimizer,
    training_state,
    dataset,
    summary_writer,
    epochs_per_test=5
)

100%|██████████| 500/500 [02:51<00:00,  2.91it/s]
Epoch 1 Train Loss=1.53e+00, BER=4.80e-02, Time=171.96s
100%|██████████| 500/500 [02:49<00:00,  2.94it/s]
Epoch 2 Train Loss=9.20e-01, BER=2.83e-02, Time=169.84s
100%|██████████| 500/500 [02:51<00:00,  2.91it/s]
Epoch 3 Train Loss=7.98e-01, BER=2.18e-02, Time=171.69s
100%|██████████| 500/500 [02:49<00:00,  2.95it/s]
Epoch 4 Train Loss=7.51e-01, BER=1.97e-02, Time=169.53s
100%|██████████| 500/500 [02:49<00:00,  2.95it/s]
Epoch 5 Train Loss=7.08e-01, BER=1.80e-02, Time=169.68s
100%|██████████| 500/500 [00:14<00:00, 34.90it/s]


KeyError: 'overall_ber'